# Get Default GCP Credentials

In [5]:
import datetime

import google.auth.impersonated_credentials
import google.auth.transport.requests
from google.cloud import secretmanager

class GCPAuth:
    def __init__(self, project_id: str, impersonate_service_account: str = None, lifetime: int = 3600):
        """
        Initializes the GCPAuth class.
        """
        self.project_id = project_id
        self.impersonate_service_account = impersonate_service_account
        self.lifetime = lifetime
        self.creds = None

    def get_remaining_lifetime(self):
        """        
        Returns the remaining lifetime of the credentials in seconds.
        If the credentials are None, returns 0.
        If the credentials are expired, returns 0.
        If the credentials are valid, returns the remaining lifetime in seconds.
        """
        if self.creds is not None:
            now = datetime.datetime.now(datetime.timezone.utc).timestamp()
            expiration = self.creds.expiry.replace(
                tzinfo=datetime.timezone.utc).timestamp()
            return max(int(expiration) - int(now), 0)
        return 0

    def is_token_expired(self):
        """       
        Checks if the token is expired.
        Returns True if the token is expired, False otherwise.
        """
        remaining_lifetime = self.get_remaining_lifetime()
        return remaining_lifetime == 0

    @staticmethod
    def access_secret_version(creds, project_id, secret_id, version_id="latest"):
        client = secretmanager.SecretManagerServiceClient(
            credentials=creds)
        name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
        response = client.access_secret_version(request={"name": name})
        return response.payload.data.decode('UTF-8')

    def get_credentials(self):
        """
        Retrieves the GCP credentials.
        If the credentials are already cached and not expired, returns them.
        If the credentials are not cached or expired, authenticates and returns new credentials.
        """
        if self.creds is not None and not self.is_token_expired():
            # logger.info("Using cached credentials")
            print("Using cached credentials")
            return self.creds

        if self.creds is None:
            # logger.info("No credentials found, authenticating...")
            print("No credentials found, authenticating...")
            target_scopes = [
                "https://www.googleapis.com/auth/cloud-platform"
            ]
            creds, pid = google.auth.default(scopes=target_scopes)
            if self.impersonate_service_account is not None:
                self.creds = google.auth.impersonated_credentials.Credentials(
                    source_credentials=creds,
                    target_principal=self.impersonate_service_account,
                    target_scopes=target_scopes,
                    lifetime=self.lifetime,  # seconds up to 3600 (1h)
                )
                # logger.info(f"Authenticated with SA {self.impersonate_service_account}")
                print(
                    f"Authenticated with SA {self.impersonate_service_account}")
            else:
                self.creds = creds
                if hasattr(self.creds, "service_account_email"):
                    print(self.creds.service_account_email)
                else:
                    print("Authenticated with default credentials ")
        # logger.info("Refreshing credentials...")
        print("Refreshing credentials...")
        self.creds.refresh(google.auth.transport.requests.Request())
        return self.creds

In [6]:
# Get credentials
auth = GCPAuth(project_id="ingka-map-services-dev")
CREDENTIALS = auth.get_credentials()

No credentials found, authenticating...


/opt/anaconda3/envs/SP/lib/python3.11/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Authenticated with default credentials 
Refreshing credentials...


# Get model  - Langchain

In [7]:
from typing import Union

from google.auth.impersonated_credentials import Credentials
from langchain_google_vertexai import ChatVertexAI
from langchain_google_vertexai.model_garden import ChatAnthropicVertex


def get_model(location: str,
              project_id: str,
              model_name: str,
              temperature: float = 0.0,
              gcp_credentials: Credentials = None) -> Union[None | ChatVertexAI | ChatAnthropicVertex]:
    """
    Returns a model instance based on the provided parameters.
    """

    if model_name.startswith("claude"):
        if gcp_credentials is None:
            raise ValueError(
                "GCP credentials must be provided for Anthropic models.")
        return ChatAnthropicVertex(
            model_name=model_name,
            project=project_id,
            location=location,
            api_transport="rest",
            temperature=temperature,
            # by passing credentials as attribute, every invoke renews the token, even if it's not expired
            credentials=gcp_credentials,
        )
    elif model_name.startswith("gemini"):
        if gcp_credentials is None:
            raise ValueError(
                "GCP credentials must be provided for Gemini models.")
        return ChatVertexAI(
            model_name=model_name,
            project=project_id,
            location=location,
            api_transport="rest",
            temperature=temperature,
            # by passing credentials as attribute, every invoke renews the token, even if it's not expired
            credentials=gcp_credentials,
        )
    else:
        raise ValueError(f"Model {model_name} not supported")


In [8]:
# Get the model
model = get_model(
    location="europe-west1",
    project_id="ingka-map-services-dev",
    model_name="gemini-2.5-flash",
    temperature=0.0,
    gcp_credentials=CREDENTIALS,
)

/var/folders/7p/n9dvhr9j5pj613b93_h9zcc80000gn/T/ipykernel_92000/1750758340.py:34: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  return ChatVertexAI(
/var/folders/7p/n9dvhr9j5pj613b93_h9zcc80000gn/T/ipykernel_92000/1750758340.py:34: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  return ChatVertexAI(


# Simple invocation - Langchain

In [9]:
# Simple invocation
response = model.invoke("What is the capital of France?")
print(response.content)

The capital of France is **Paris**.


# Official Google Gen AI and Anthropic SDKs

In [10]:
from google import genai

gemini_client = genai.Client(
    vertexai=True,
    project="ingka-map-services-dev",
    location="europe-west1",
    credentials=CREDENTIALS,
)

response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What is the capital of France?",
)
print(response.text)

The capital of France is **Paris**.


In [11]:
from anthropic import AnthropicVertex

anthropic_client = AnthropicVertex(
    region="europe-west1",
    project_id="ingka-map-services-dev",
)

response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
)
print(response.content[0].text)

/opt/anaconda3/envs/SP/lib/python3.11/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


The capital of France is **Paris**.


# Vertex AI Model Garden (OpenAI-compatible endpoint)

In [ ]:
import google.auth.transport.requests
from openai import OpenAI

PROJECT = "ingka-map-services-dev"
LOCATION = "europe-west1"

# Refresh credentials to get a valid access token
CREDENTIALS.refresh(google.auth.transport.requests.Request())

model_garden_client = OpenAI(
    base_url=f"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/projects/{PROJECT}/locations/{LOCATION}/endpoints/openapi",
    api_key=CREDENTIALS.token,
)

# Model Garden model names use the format: "provider/model-name"
# Examples:
#   "meta/llama-3.3-70b-instruct-maas"
#   "mistral-ai/mistral-nemo"
#   "google/gemma-3-27b-it"

response = model_garden_client.chat.completions.create(
    model="meta/llama-3.3-70b-instruct-maas",
    messages=[{"role": "user", "content": "What is the capital of France?"}],
    temperature=0.0,
    max_tokens=256,
)
print(response.choices[0].message.content)
print(f"\nTokens in: {response.usage.prompt_tokens}, out: {response.usage.completion_tokens}")